In [19]:
!pip install langchain langchain-text-splitters

In [20]:
import re
import json
import os

INPUT_FILE = 'DATA_WEB_FIT.md'
OUTPUT_JSON = 'CHUNKED_RECURSIVE_DATA.json'

def clean_markdown_content(text):
    # Remove author tags and breadcrumbs
    text = re.sub(r'Tác giả :\s*', '', text)
    text = re.sub(r'\[.*?\]\(/\?TopicId=.*?\)\n+', '', text)
    
    # Truncate at specific keywords (e.g., forms, editors)
    trash_keywords = ["Góp ý", "Họ và tên:", "RadEditor - HTML"]
    for kw in trash_keywords:
        if kw in text:
            text = text.split(kw)[0]
            
    # Remove specific residual image links and text
    text = re.sub(r'!\[\]\(/Resources/ImagesPortal/HomePage/tinkhac.png\)', '', text)
    text = re.sub(r'\[Các tin khác\]\(#\)', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()

In [21]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Structural chunking by Markdown headers
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# Recursive chunking fallback
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "! ",
        "? ",
        " ",
        ""
    ],
    keep_separator=True
)

In [22]:
print(f"Processing '{INPUT_FILE}'...")

try:
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        raw_data = f.read()

    parts = re.split(r'={60}\nURL: (.*?)\n={60}\n', raw_data)
    final_chunks = []
    
    for i in range(1, len(parts), 2):
        url = parts[i].strip()
        raw_content = parts[i+1].strip()
        
        clean_content = clean_markdown_content(raw_content)
        
        if clean_content:
            # Pass 1: Split by markdown headers
            md_header_splits = markdown_splitter.split_text(clean_content)
            
            # Pass 2: Recursive character split
            recursive_splits = text_splitter.split_documents(md_header_splits)
            
            # Metadata assembly
            for chunk in recursive_splits:
                meta = chunk.metadata
                meta['source_url'] = url
                
                final_chunks.append({
                    "page_content": chunk.page_content,
                    "metadata": meta
                })

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f_out:
        json.dump(final_chunks, f_out, ensure_ascii=False, indent=4)

    print(f"Success: Processed {len(parts)//2} articles into {len(final_chunks)} chunks.")
    print(f"Output saved to: {OUTPUT_JSON}")

except FileNotFoundError:
    print(f"Error: File '{INPUT_FILE}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Processing 'DATA_WEB_FIT.md'...
Success: Processed 191 articles into 421 chunks.
Output saved to: CHUNKED_RECURSIVE_DATA.json
